In [1]:
import ctypes
import numpy as np
import glfw
from OpenGL.GL import *

In [2]:
VERTEX_SHADER_SRC = """#version 330 core
layout(location = 0) in vec3 aPos;
layout(location = 1) in vec3 aColor;

uniform mat4 uModelView;
uniform mat4 uPersp;

out vec3 vPoint;   // Position in eye/camera space
out vec3 vColor;

void main() {
    vPoint = (uModelView * vec4(aPos, 1.0)).xyz;
    gl_Position = uPersp * vec4(vPoint, 1.0);
    vColor = aColor;
}
"""

FRAGMENT_SHADER_SRC = """#version 330 core
in vec3 vPoint;
in vec3 vColor;

out vec4 FragColor;

uniform vec3 uLight = vec3(2.0, 2.0, 3.0);  // Light position in eye space
uniform float uAmb = 0.15;                  // Ambient reflection weight
uniform float uDif = 0.75;                  // Diffuse reflection weight
uniform float uSpc = 0.60;                  // Specular reflection weight

void main() {
    // 1. Analytical face normal via screen-space derivatives (faceted appearance)
    vec3 dx = dFdx(vPoint);
    vec3 dy = dFdy(vPoint);
    vec3 N = normalize(cross(dx, dy));

    // 2. Diffuse term (Lambertian)
    vec3 L = normalize(uLight - vPoint);
    float d = max(0.0, dot(N, L));

    // 3. Specular highlight (Phong reflection)
    vec3 E = normalize(-vPoint);             // Eye vector toward origin in eye space
    vec3 R = reflect(-L, N);
    float h = max(0.0, dot(R, E));
    float s = pow(h, 64.0);

    // 4. Combined illumination
    float intensity = min(1.0, uAmb + uDif * d) + uSpc * s;
    FragColor = vec4(intensity * vColor, 1.0);
}
"""

In [3]:
NUM_FRONT = 10           # vertices per face (front 0..9, back 10..19)
Z_FRONT, Z_BACK = 0.10, -0.10


def build_extruded_letter_n():
    front_2d = np.array([
        [  0.0,   0.0],  # 0 A
        [ 60.0,   0.0],  # 1 B
        [ 60.0, 210.0],  # 2 C
        [240.0,   0.0],  # 3 D
        [300.0,   0.0],  # 4 E
        [300.0, 300.0],  # 5 F
        [240.0, 300.0],  # 6 G
        [240.0,  90.0],  # 7 H
        [ 60.0, 300.0],  # 8 I
        [  0.0, 300.0],  # 9 J
    ], dtype=np.float32)

    # Center the bounding box on (0,0) and scale uniformly so the 300-unit height spans [-0.8, 0.8]
    center_xy = (front_2d.max(axis=0) + front_2d.min(axis=0)) / 2.0
    front_2d = (front_2d - center_xy) * (1.6 / 300.0)

    # 1. Vertices: front copy at z=+0.10, back copy at z=-0.10
    front_verts = np.column_stack([front_2d, np.full(NUM_FRONT, Z_FRONT, dtype=np.float32)])
    back_verts = np.column_stack([front_2d, np.full(NUM_FRONT, Z_BACK, dtype=np.float32)])
    vertices = np.vstack([front_verts, back_verts]).astype(np.float32)

    # 2. One distinct color per outline vertex (same palette on front and back)
    palette = np.array([
        [0.0, 0.0, 1.0],  # A blue
        [1.0, 1.0, 0.0],  # B yellow
        [1.0, 0.5, 0.0],  # C orange
        [1.0, 0.0, 1.0],  # D magenta
        [0.0, 1.0, 1.0],  # E cyan
        [0.0, 1.0, 0.0],  # F green
        [0.5, 0.0, 1.0],  # G violet
        [1.0, 0.0, 0.5],  # H pink
        [1.0, 0.0, 0.0],  # I red
        [0.0, 0.6, 0.3],  # J teal
    ], dtype=np.float32)
    colors = np.vstack([palette, palette])

    # 3a. Front face: the 8 counter-clockwise HW3 triangles (normal = +z)
    front_tris = [
        [0, 1, 2], [0, 2, 9], [2, 8, 9],   # left stem
        [2, 3, 7], [2, 7, 8],              # diagonal band
        [3, 4, 7], [7, 4, 5], [7, 5, 6],   # right stem
    ]

    # 3b. Back face (TODO 1): offset by +10 and swap the last two indices to reverse winding (normal = -z)
    back_tris = [[a + NUM_FRONT, c + NUM_FRONT, b + NUM_FRONT] for a, b, c in front_tris]

    # 3c. Side walls (TODO 2): one quad per directed outline edge i -> j, split into two triangles.
    #     Outline is CCW, so the outward side of edge i->j is its right-hand side; these windings
    #     make cross(v1-v0, v2-v0) point that way (see the winding note above).
    boundary_cycle = list(range(NUM_FRONT)) + [0]   # A B C D E F G H I J A
    side_tris = []
    for i, j in zip(boundary_cycle[:-1], boundary_cycle[1:]):
        i_back, j_back = i + NUM_FRONT, j + NUM_FRONT
        side_tris.append([i, i_back, j])
        side_tris.append([j, i_back, j_back])

    indices = np.array(front_tris + back_tris + side_tris, dtype=np.uint32).flatten()
    return vertices, colors, indices

In [4]:
def _point_in_polygon(p, poly):
    """Even-odd ray casting test for a 2D point."""
    x, y = p
    inside = False
    for k in range(len(poly)):
        (x1, y1), (x2, y2) = poly[k], poly[(k + 1) % len(poly)]
        if (y1 > y) != (y2 > y) and x < x1 + (y - y1) * (x2 - x1) / (y2 - y1):
            inside = not inside
    return inside


def check_outward_topology(vertices, indices, eps=1e-3):
    tris = indices.reshape(-1, 3)
    outline = vertices[:NUM_FRONT, :2]
    for t in tris:
        v0, v1, v2 = vertices[t]
        n = np.cross(v1 - v0, v2 - v0)
        assert np.linalg.norm(n) > 1e-9, f"degenerate triangle {t}"
        p = (v0 + v1 + v2) / 3.0 + eps * n / np.linalg.norm(n)
        inside = Z_BACK < p[2] < Z_FRONT and _point_in_polygon(p[:2], outline)
        assert not inside, f"triangle {t.tolist()} faces inward"

    # Closed (watertight) mesh: each undirected edge used exactly twice, once in each direction
    directed = [(t[k], t[(k + 1) % 3]) for t in tris for k in range(3)]
    assert len(set(directed)) == len(directed), "duplicate directed edge (inconsistent winding)"
    assert all((b, a) in set(directed) for a, b in directed), "mesh is not closed"
    print(f"OK: {len(tris)} triangles (8 front + 8 back + 20 side), all outward, mesh closed.")


_v, _c, _i = build_extruded_letter_n()
check_outward_topology(_v, _i)

OK: 36 triangles (8 front + 8 back + 20 side), all outward, mesh closed.


In [5]:
class ArcballCamera:
    SENSITIVITY = 0.4  # degrees per pixel

    def __init__(self, distance=3.0):
        self.distance = distance
        self.rot_x = 20.0   # Pitch (degrees)
        self.rot_y = -35.0  # Yaw (degrees)
        self.last_x = 0.0
        self.last_y = 0.0
        self.is_dragging = False

    def on_mouse_press(self, x, y):
        self.is_dragging = True
        self.last_x = x
        self.last_y = y

    def on_mouse_release(self):
        self.is_dragging = False

    def on_mouse_drag(self, x, y):
        if not self.is_dragging:
            return
        dx = x - self.last_x
        dy = y - self.last_y
        self.last_x = x
        self.last_y = y

        # TODO 4: horizontal drag -> yaw, vertical drag -> pitch (clamped to avoid flipping)
        self.rot_y += dx * self.SENSITIVITY
        self.rot_x = float(np.clip(self.rot_x + dy * self.SENSITIVITY, -89.0, 89.0))

    def get_modelview(self):
        rad_x = np.radians(self.rot_x)
        rad_y = np.radians(self.rot_y)

        # Rotation around X (Pitch)
        Rx = np.array([
            [1.0, 0.0,            0.0,           0.0],
            [0.0, np.cos(rad_x), -np.sin(rad_x), 0.0],
            [0.0, np.sin(rad_x),  np.cos(rad_x), 0.0],
            [0.0, 0.0,            0.0,           1.0]
        ], dtype=np.float32)

        # Rotation around Y (Yaw)
        Ry = np.array([
            [ np.cos(rad_y), 0.0, np.sin(rad_y), 0.0],
            [ 0.0,           1.0, 0.0,           0.0],
            [-np.sin(rad_y), 0.0, np.cos(rad_y), 0.0],
            [ 0.0,           0.0, 0.0,           1.0]
        ], dtype=np.float32)

        # Translation back along Z
        T = np.identity(4, dtype=np.float32)
        T[2, 3] = -self.distance

        # TODO 5: V = T * Rx(pitch) * Ry(yaw); yaw is applied to the model first, translation last
        return T @ Rx @ Ry


def make_perspective(fov_rad, aspect, near, far):
    tan_half_fov = np.tan(fov_rad / 2.0)
    res = np.zeros((4, 4), dtype=np.float32)
    res[0, 0] = 1.0 / (aspect * tan_half_fov)
    res[1, 1] = 1.0 / tan_half_fov
    res[2, 2] = -(far + near) / (far - near)
    res[2, 3] = -(2.0 * far * near) / (far - near)
    res[3, 2] = -1.0
    return res

In [6]:
def compile_shader(src, shader_type):
    s = glCreateShader(shader_type)
    glShaderSource(s, src)
    glCompileShader(s)
    if not glGetShaderiv(s, GL_COMPILE_STATUS):
        raise RuntimeError(glGetShaderInfoLog(s).decode())
    return s


def build_program():
    vs = compile_shader(VERTEX_SHADER_SRC, GL_VERTEX_SHADER)
    fs = compile_shader(FRAGMENT_SHADER_SRC, GL_FRAGMENT_SHADER)
    prog = glCreateProgram()
    glAttachShader(prog, vs)
    glAttachShader(prog, fs)
    glLinkProgram(prog)
    if not glGetProgramiv(prog, GL_LINK_STATUS):
        raise RuntimeError(glGetProgramInfoLog(prog).decode())
    glDeleteShader(vs)
    glDeleteShader(fs)
    return prog

In [7]:
def main():
    """Open the window and run the interactive render loop until it is closed."""
    if not glfw.init():
        raise RuntimeError("Failed to initialize GLFW.")

    glfw.window_hint(glfw.CONTEXT_VERSION_MAJOR, 3)
    glfw.window_hint(glfw.CONTEXT_VERSION_MINOR, 3)
    glfw.window_hint(glfw.OPENGL_PROFILE, glfw.OPENGL_CORE_PROFILE)
    glfw.window_hint(glfw.OPENGL_FORWARD_COMPAT, GL_TRUE)  # Required for macOS

    window = glfw.create_window(800, 700, "HW4: 3D Letter N - Shading and Camera", None, None)
    if not window:
        glfw.terminate()
        raise RuntimeError("Failed to create GLFW window.")

    glfw.make_context_current(window)
    try:
        render_loop(window)
    finally:
        # Always release the window, even after an error, so it never stays open frozen
        glfw.destroy_window(window)
        glfw.terminate()


def render_loop(window):
    """Set up GL state, callbacks and buffers, then draw until the window is closed."""
    # TODO 3a (Visibility): enable Z-buffer depth testing
    glEnable(GL_DEPTH_TEST)
    glDepthFunc(GL_LESS)

    camera = ArcballCamera(distance=2.8)

    # GLFW Mouse Callbacks
    def mouse_button_cb(w, button, action, mods):
        if button == glfw.MOUSE_BUTTON_LEFT:
            if action == glfw.PRESS:
                x, y = glfw.get_cursor_pos(w)
                camera.on_mouse_press(x, y)
            elif action == glfw.RELEASE:
                camera.on_mouse_release()

    def cursor_pos_cb(w, xpos, ypos):
        camera.on_mouse_drag(xpos, ypos)

    glfw.set_mouse_button_callback(window, mouse_button_cb)
    glfw.set_cursor_pos_callback(window, cursor_pos_cb)

    # Build geometry and upload it to the GPU
    verts, colors, indices = build_extruded_letter_n()

    vao = glGenVertexArrays(1)
    glBindVertexArray(vao)

    # Interleave attributes: [x, y, z, r, g, b]
    interleaved = np.empty((len(verts), 6), dtype=np.float32)
    interleaved[:, 0:3] = verts
    interleaved[:, 3:6] = colors

    vbo = glGenBuffers(1)
    glBindBuffer(GL_ARRAY_BUFFER, vbo)
    glBufferData(GL_ARRAY_BUFFER, interleaved.nbytes, interleaved, GL_STATIC_DRAW)

    ebo = glGenBuffers(1)
    glBindBuffer(GL_ELEMENT_ARRAY_BUFFER, ebo)
    glBufferData(GL_ELEMENT_ARRAY_BUFFER, indices.nbytes, indices, GL_STATIC_DRAW)

    stride = 6 * interleaved.itemsize
    glVertexAttribPointer(0, 3, GL_FLOAT, GL_FALSE, stride, ctypes.c_void_p(0))      # position
    glEnableVertexAttribArray(0)
    glVertexAttribPointer(1, 3, GL_FLOAT, GL_FALSE, stride, ctypes.c_void_p(3 * 4))  # color
    glEnableVertexAttribArray(1)

    glBindVertexArray(0)

    prog = build_program()
    loc_mv = glGetUniformLocation(prog, "uModelView")
    loc_pj = glGetUniformLocation(prog, "uPersp")

    while not glfw.window_should_close(window):
        w, h = glfw.get_framebuffer_size(window)
        glViewport(0, 0, w, h)
        aspect = w / float(h if h > 0 else 1)

        glClearColor(0.12, 0.14, 0.18, 1.0)
        # TODO 3b (Visibility): clear BOTH the color and the depth buffer every frame
        glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

        MV = camera.get_modelview()
        P = make_perspective(np.radians(40.0), aspect, 0.1, 50.0)

        glUseProgram(prog)
        # numpy is row-major, GLSL expects column-major -> upload the transpose
        glUniformMatrix4fv(loc_mv, 1, GL_FALSE, MV.T.astype(np.float32))
        glUniformMatrix4fv(loc_pj, 1, GL_FALSE, P.T.astype(np.float32))

        glBindVertexArray(vao)
        glDrawElements(GL_TRIANGLES, len(indices), GL_UNSIGNED_INT, None)
        glBindVertexArray(0)

        glfw.swap_buffers(window)
        glfw.poll_events()

    glDeleteVertexArrays(1, [vao])
    glDeleteBuffers(1, [vbo])
    glDeleteBuffers(1, [ebo])
    glDeleteProgram(prog)

In [ ]:
if __name__ == "__main__":
    main()

: 